### **SECTION 1 — SOURCE TO TARGET TESTING**

In [0]:
%sql
-- Row Count Validation

SELECT COUNT(*) AS bronze_sales_count
FROM retail_lakehouse.bronze.sales;

SELECT COUNT(*) AS silver_sales_count
FROM retail_lakehouse.silver.sales;

SELECT COUNT(*) AS gold_fact_count
FROM retail_lakehouse.gold.fact_sales;


In [0]:
%sql
-- Column Mapping Validation
SELECT
    b.CustomerID AS bronze_customer_id,
    s.CustomerID AS silver_customer_id,
    b.CustomerName AS bronze_customer_name,
    s.CustomerName AS silver_customer_name
FROM retail_lakehouse.bronze.customers b
JOIN retail_lakehouse.silver.customers s
ON b.CustomerID = s.CustomerID
LIMIT 10;

In [0]:
%sql
-- Data Type Validation
DESCRIBE retail_lakehouse.silver.sales;
DESCRIBE retail_lakehouse.gold.fact_sales;

### **SECTION 2 — DATA TRANSFORMATION TESTING**

In [0]:
%sql
-- Proper Case Validation
SELECT CustomerName
FROM retail_lakehouse.silver.customers
WHERE CustomerName != INITCAP(CustomerName);

In [0]:
%sql
-- Lowercase Validation
SELECT Email
FROM retail_lakehouse.silver.customers
WHERE Email != LOWER(Email);

In [0]:
%sql
-- Date Formatting Validation
SELECT TxnDate
FROM retail_lakehouse.silver.sales
LIMIT 10;

In [0]:
%sql
-- Derived Amount Validation
SELECT
    s.TransactionID,
    s.Quantity,
    p.UnitPrice,
    (s.Quantity * p.UnitPrice) AS ExpectedAmount,
    f.Amount AS ActualAmount
FROM retail_lakehouse.gold.fact_sales f

JOIN retail_lakehouse.silver.sales s
ON f.TransactionID = s.TransactionID

JOIN retail_lakehouse.silver.products p
ON s.ProductID = p.ProductID
LIMIT 10;

### **SECTION 3 — DATA QUALITY TESTING**

In [0]:
%sql
-- Duplicate Validation
SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.silver.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Null Validation
SELECT *
FROM retail_lakehouse.silver.sales
WHERE TransactionID IS NULL;

In [0]:
%sql
-- Invalid Data Validation
SELECT *
FROM retail_lakehouse.silver.sales
WHERE Quantity <= 0;

In [0]:
%sql
SELECT *
FROM retail_lakehouse.silver.products
WHERE UnitPrice <= 0;

### **SECTION 4 — REFERENTIAL INTEGRITY TESTING**

In [0]:
%sql
-- Customer Referential Integrity
SELECT *
FROM retail_lakehouse.silver.sales s

LEFT JOIN retail_lakehouse.gold.dim_customer c
ON s.CustomerID = c.CustomerID

WHERE c.CustomerID IS NULL;

In [0]:
%sql
-- Product Referential Integrity
SELECT *
FROM retail_lakehouse.silver.sales s

LEFT JOIN retail_lakehouse.gold.dim_product p
ON s.ProductID = p.ProductID

WHERE p.ProductID IS NULL;

In [0]:
%sql
-- Store Referential Integrity
SELECT *
FROM retail_lakehouse.silver.sales s

LEFT JOIN retail_lakehouse.gold.dim_store st
ON s.StoreID = st.StoreID

WHERE st.StoreID IS NULL;

### **SECTION 5 — SCD TYPE 2 VALIDATION**

In [0]:
%sql
-- Active vs Inactive Validation
SELECT
    CustomerID,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate;

In [0]:
%sql
-- Only One Active Record Validation
SELECT
    CustomerID,
    COUNT(*)
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

### **SECTION 6 — CDC VALIDATION**

In [0]:
%sql
SELECT
    _change_type,
    _commit_version,
    _commit_timestamp,
    TransactionID
FROM table_changes(
    'retail_lakehouse.silver.sales',
    1
)
ORDER BY _commit_version DESC;

### **SECTION 7 — FULL LOAD vs INCREMENTAL LOAD**

In [0]:
%sql
-- Full Load Validation
SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;

In [0]:
%sql
-- Incremental Load Validation
SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC;